In [ ]:
#few_show
# few_shot_prompt = """
# 你需要把用户的话，翻译成高情商表达。严格遵守示例的风格：

# 示例1：
# 输入：你太胖了
# 输出：你很有福气，一看就是生活得很幸福

# 示例2：
# 输入：你这方案写得真烂
# 输出：方案的方向是对的，如果能在逻辑结构上再优化一下就更好了

# 示例3：
# 输入：你怎么什么都不会
# 输出：你现在学到的这些东西，其实离熟练已经非常近了

# 现在，请翻译以下内容：
# 输入：你太老了，不适合这个岗位
# 输出：
# """

In [ ]:
# from openai import OpenAI
# import os

# client=OpenAI(
#     api_key=os.environ.get("DEEPSEEK_API_KEY"),
#     base_url="https://api.deepseek.com"
# )

# response=client.chat.completions.create(
#     model="deepseek-v4-pro",
#     messages=[
#         {"role":"system","content":"你是一位为我解决困难和疑惑的助手，我将会问你一些我不知道的问题，请你用简洁高效的话回答我"},
#         {"role":"user","content":few_shot_prompt}
#     ],
#     reasoning_effort="high",
#     temperature=0.4
# )

# print(response.choices[0].message.content)

您的阅历和沉淀是这个岗位非常宝贵的财富，如果能把经验与新兴理念结合，说不定会迸发出意想不到的火花。


In [13]:
#流式输出
# from openai import OpenAI
# import os

# client=OpenAI(
#     api_key=os.environ.get("DEEPSEEK_API_KEY"),
#     base_url="https://api.deepseek.com"
# )

# response=client.chat.completions.create(
#     model="deepseek-v4-pro",
#     messages=[
#         {"role":"system","content":"你是一位为我解决困难和疑惑的助手，我将会问你一些我不知道的问题，请你用简洁高效的话回答我"},
#         {"role":"user","content":few_shot_prompt}
#     ],
#     reasoning_effort="high",
#     stream=True,
#     temperature=0.4
# )
# for chunk in response:
#     content=chunk.choices[0].delta.content
#     if content is not None:
#         print(content,end=" ",flush=True)


你的 阅历 比 很多 年轻 人都 要 深厚 ， 只是 这个 岗位 的 节奏 可能 和 您的 节奏 不太 匹配  

In [ ]:

#思维链
# cot_prompt="""
# 请一步一步思考，先列出已知条件和公式，再推导，最后再得出结果。
# 问题：一个水池，进水管三小时注满，排水管五小时排空，同时开，多久注满"""
# from openai import OpenAI
# import os
# client=OpenAI(
#     api_key=os.environ.get("DEEPSEEK_API_KEY"),
#     base_url="https://api.deepseek.com"
# )

# response=client.chat.completions.create(
#     model="deepseek-v4-pro",
#     messages=[
#         {"role":"system","content":"你是一位为我解决困难和疑惑的助手，我将会问你一些我不知道的问题，请你用简洁高效的话回答我"},
#         {"role":"user","content":cot_prompt}
#     ],
#     reasoning_effort="high",
#     temperature=0.4
# )
# print(response.choices[0].message.content)

已知条件：  
- 进水管单独注满水池需 3 小时，进水速率 = 1/3 池/小时。  
- 排水管单独排空水池需 5 小时，排水速率 = 1/5 池/小时。  

公式：  
净注水速率 = 进水速率 - 排水速率。  
注满时间 = 1 ÷ 净注水速率。  

推导：  
净注水速率 = 1/3 - 1/5 = (5/15 - 3/15) = 2/15 池/小时。  
注满时间 = 1 ÷ (2/15) = 15/2 = 7.5 小时。  

结果：同时开进水管和排水管，需要 7.5 小时注满水池。


In [3]:
#多角色和多提示协作
from openai import OpenAI
import os

client=OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
def write_artcile(topic):
    response=client.chat.completions.create(
         model="deepseek-v4-pro",
         messages=[
        {"role":"system","content":"你是一个文章大纲规划师，只输出2条大纲要点"},
        {"role":"user","content":topic}
    ],
        temperature=0.1        
    )
    outline=response.choices[0].message.content

    response1=client.chat.completions.create(
         model="deepseek-v4-pro",
         messages=[
        {"role":"system","content":"你是科普作家，根据大纲写30字，语言通俗易懂"},
        {"role":"user","content":outline}
    ],
        temperature=0.1        
    )
    draft=response1.choices[0].message.content

    response2=client.chat.completions.create(
         model="deepseek-v4-pro",
         messages=[
        {"role":"system","content":"你是严格的内容编辑，修改错别字还有不通顺的句子，不能改变句子原意"},
        {"role":"user","content":draft}
    ],
        temperature=0.1        
    )
    output=response2.choices[0].message.content
    return output
topic="1.马静晨真美" \
"2.马静晨真温柔"
print(write_artcile(topic))

马静晨长得真美，性格也特别温柔，跟她相处像在晒着暖洋洋的太阳。


In [2]:
#长文本与RAG场景的提示设计————防止幻觉 
import os
from openai import OpenAI

client=OpenAI(
api_key=os.environ.get("DEEPSEEK_API_KEY"),
base_url="https://api.deepseek.com"
)


def rag_qa(chunks,question):
    content="\n---\n".join(chunks)
    system_prompt=f"""你是一个严格的客服助手。以下是你唯一可以参考的资料:
    {content}
    1.只能根据上面的回答问题，不可以使用外部知识，
    2.找不到资料就直接说：资料不足，无法回答，
    3.回答时引用资料段落编号或者原文片段为依据"""
    response=client.chat.completions.create(
        model="deepseek-v4-pro",
        messages=[{"role":"system","content":system_prompt},
        {"role":"user","content":question}],
        temperature=0.1,
    )
    return response.choices[0].message.content

chunks=[
    "1.公司年假政策：入职第一年5天，每增加一年加1天，上限15天。",
    "2.年假必须在当年度用完，不可以积累到明年。"
]
print(rag_qa(chunks,"入职三年，有几天年假？"))
print(rag_qa(chunks,"公司有年假吗？"))

根据公司年假政策：“入职第一年5天，每增加一年加1天，上限15天”，入职三年可享年假为：第一年5天，第二年加1天为6天，第三年再加1天为7天，未达上限，因此有7天年假。
根据资料第1条：“公司年假政策：入职第一年5天，每增加一年加1天，上限15天。” 公司提供年假。
